# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [37]:
print("One row = one daily observation for client x article x day")
print("Tables: fact_content_daily_performance (main), dim_content and dim_clients for names/joins only")
print("Time window: Feature window = 2026-03-01 to 2026-03-31 (March). Target window = 2026-04-01 to 2026-04-30 (April)")
print("Label/Proxy: PROXY - declining if April impressions < 80% of March impressions, with March impressions >= 100")
print("Deliberate Exclusion: any feature computed from April data (the target window)")

One row = one daily observation for client x article x day
Tables: fact_content_daily_performance (main), dim_content and dim_clients for names/joins only
Time window: Feature window = 2026-03-01 to 2026-03-31 (March). Target window = 2026-04-01 to 2026-04-30 (April)
Label/Proxy: PROXY - declining if April impressions < 80% of March impressions, with March impressions >= 100
Deliberate Exclusion: any feature computed from April data (the target window)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [38]:
features = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_days_with_impressions"
]

label_proxy = "decline_proxy = 1 if april_impressions < 0.8 * march_impressions and march_impressions >= 100 else 0"

context = ["client_hash_id", "content_hash_id", "feature_month", "target_month"]

excluded = {
    "april_impressions": "target-window outcome; cannot be a feature",
    "any_april_metric_feature": "future leakage",
    "client_hash_id": "ID for grouping/splits, not as predictive feature",
    "content_hash_id": "ID for joining/tracking, not as predictive feature"
}

print("FEATURES:", features)
print("LABEL/PROXY:", label_proxy)
print("CONTEXT:", context)
print("EXCLUDED:")
for k, v in excluded.items():
    print(f"- {k}: {v}")

FEATURES: ['march_impressions', 'march_clicks', 'march_ctr', 'march_avg_position', 'march_days_with_impressions']
LABEL/PROXY: decline_proxy = 1 if april_impressions < 0.8 * march_impressions and march_impressions >= 100 else 0
CONTEXT: ['client_hash_id', 'content_hash_id', 'feature_month', 'target_month']
EXCLUDED:
- april_impressions: target-window outcome; cannot be a feature
- any_april_metric_feature: future leakage
- client_hash_id: ID for grouping/splits, not as predictive feature
- content_hash_id: ID for joining/tracking, not as predictive feature


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [39]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [40]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [41]:
# 1. Grain Check
con.sql(f"""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT client_hash_id || content_hash_id) as unique_keys
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03*/*.parquet')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────┐
│ total_rows │ unique_keys │
│   int64    │    int64    │
├────────────┼─────────────┤
│    9841378 │      331437 │
└────────────┴─────────────┘



In [42]:
# 2) Counts: rows eligible by March monthly impressions >= 100
con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03*/*.parquet')
    GROUP BY 1,2
)
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN march_impressions >= 100 THEN 1 ELSE 0 END) AS eligible_rows
FROM march
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┐
│ total_rows │ eligible_rows │
│   int64    │    int128     │
├────────────┼───────────────┤
│     331437 │        101441 │
└────────────┴───────────────┘



In [43]:
# 3) Missing values in March monthly features
con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        AVG(gsc_clicks) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03*/*.parquet')
    GROUP BY 1,2
)
SELECT
    SUM(CASE WHEN march_clicks IS NULL THEN 1 ELSE 0 END) AS null_clicks,
    SUM(CASE WHEN march_avg_position IS NULL THEN 1 ELSE 0 END) AS null_avg_position,
    SUM(CASE WHEN march_impressions = 0 THEN 1 ELSE 0 END) AS zero_impressions
FROM march
""").show()

┌─────────────┬───────────────────┬──────────────────┐
│ null_clicks │ null_avg_position │ zero_impressions │
│   int128    │      int128       │      int128      │
├─────────────┼───────────────────┼──────────────────┤
│           0 │            154699 │           154699 │
└─────────────┴───────────────────┴──────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


- **Two-month window only**: this contract is built on March→April 2026 alone. It cannot speak to seasonality, longer-term trends, or whether a decline is a one-off dip vs. a sustained pattern.
- **Unbalanced history across clients/content**: some `client_hash_id` / `content_hash_id` pairs may have far more historical data than others, so the model may learn patterns that only hold for well-represented clients.
- **GSC-only early rows**: rows sourced only from Google Search Console (before other integrations came online) may have different measurement definitions or coverage than later rows, making early data less comparable to recent data.
- **Window overlap risk**: if "March" and "April" boundaries aren't strictly non-overlapping (e.g. due to timezone cutoffs or reporting lag), some days could be double-counted or missing, silently biasing the impressions/clicks totals.
- **Threshold artifact**: the `march_impressions >= 100` cutoff excludes low-traffic content entirely — the proxy label says nothing about decline behavior for low-volume pages.
- **Proxy label, not ground truth**: `decline_proxy` is a heuristic (20% drop), not a verified business definition of "decline" — it may mislabel normal week-to-week variance as decline, or miss slower multi-month erosion.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.